# Bengali ASR — 90%+ accuracy attempt (KAGGLE version)

## ⚠️ BEFORE YOU RUN — two settings you MUST turn on
Open the **⋮ (right sidebar) → Settings**:
1. **Accelerator → `GPU P100`** (or `GPU T4 x2`)  ← without this it will crawl on CPU forever
2. **Internet → `On`**  ← needed to download the model + dataset

*(Internet requires a phone-verified Kaggle account — free, takes 1 min.)*

## Why Kaggle beats free Colab here
| | Free Colab | **Kaggle** |
|---|---|---|
| GPU hours | vague, throttled | **30 hrs/week guaranteed** |
| Session length | disconnects often | **up to 12 hours** |
| Our 10-hr job | needs 2-3 sessions | **fits in ONE session** ✅ |

## The plan
- **Dataset:** OpenSLR SLR53 — clean, **READ** Bengali (229 hrs available). This is the key: our old data (Vaani) was *spontaneous conversation*, where even papers only reach 24–31% WER. Clean read speech is the only place <10% WER exists.
- **Model:** `whisper-large-v3` + LoRA.
- **Epochs:** 3 (NOT 110 — with a big dataset, few epochs is correct; 110 would overfit and make the honest score *worse*).
- **Checkpoints:** every 200 steps to `/kaggle/working` → survives interruptions.

## Honest expectation
- **Character accuracy: 90%+ very likely.**
- **Word accuracy: likely 85–90%; 90%+ is possible but NOT guaranteed.**
- Both measured on clips the model **never trained on**.

In [ ]:
# 1. CONFIRM WE HAVE A GPU  (if this fails -> Settings -> Accelerator -> GPU P100)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU P100, then re-run."
print("GPU OK ->", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install deps  (needs Internet = On in Settings)
!pip install -q --upgrade transformers peft accelerate jiwer librosa soundfile

# some images ship an old torchao that newer peft rejects -> we don't use it, remove it
!pip uninstall -y -q torchao 2>/dev/null

print("deps ready")

In [ ]:
# 3. CONFIG  (sized to finish inside a 12-hour Kaggle session)
import os, torch

CKPT_DIR = "/kaggle/working/ckpt"        # persists as notebook Output
os.makedirs(CKPT_DIR, exist_ok=True)

MODEL    = "openai/whisper-large-v3"      # strongest Bengali. OOM? -> "openai/whisper-medium"
N_PARTS  = 4      # SLR53 has 10 parts (~1.5GB each). 4 ~= 76k clips. Raise for more accuracy.
EPOCHS   = 3      # correct for a large dataset. Do NOT set 110 - it would overfit.
BATCH    = 4      # fits P100/T4 16GB with large-v3
GRAD_ACC = 4      # effective batch = 16
LR       = 1e-3
N_EVAL   = 500    # held-out clips for the honest score
MAX_SEC  = 20

print(f"{MODEL} | parts={N_PARTS} | epochs={EPOCHS} | batch={BATCH}x{GRAD_ACC}")
print(f"checkpoints -> {CKPT_DIR}")

In [ ]:
# 4. Download OpenSLR SLR53 (clean READ Bengali). Public, no auth. ~1.5GB per part.
BASE = "https://www.openslr.org/resources/53"

!wget -q -nc {BASE}/utt_spk_text.tsv

for i in range(N_PARTS):
    z = f"asr_bengali_{i}.zip"
    if not os.path.exists(z):
        print(f"downloading {z} ...")
        !wget -q {BASE}/{z}
    print(f"unzipping {z} ...")
    !unzip -q -n {z} -d /kaggle/temp/slr53

print("download done")

In [ ]:
# 5. Build (audio, text) pairs + an honest held-out split
import glob, random

utt2text = {}
with open("utt_spk_text.tsv", encoding="utf-8") as f:
    for line in f:
        p = line.rstrip("\n").split("\t")
        if len(p) >= 3:
            utt2text[p[0]] = p[2].strip()

files = glob.glob("/kaggle/temp/slr53/**/*.flac", recursive=True)
pairs = []
for fp in files:
    t = utt2text.get(os.path.basename(fp).replace(".flac", ""))
    if t:
        pairs.append({"path": fp, "text": t})

random.seed(0); random.shuffle(pairs)
eval_pairs, train_pairs = pairs[:N_EVAL], pairs[N_EVAL:]
print(f"TRAIN {len(train_pairs)}  |  HELD-OUT {len(eval_pairs)} (never trained on)")

In [ ]:
# 6. Dataset (lazy loading -> low RAM)
import librosa, numpy as np
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL, language="bengali", task="transcribe")

class SLR53(torch.utils.data.Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        p = self.pairs[i]
        wav, _ = librosa.load(p["path"], sr=16000)
        wav = wav[: MAX_SEC * 16000].astype(np.float32)
        return {"input_features": processor.feature_extractor(wav, sampling_rate=16000).input_features[0],
                "labels": processor.tokenizer(p["text"]).input_ids[:448]}

train_ds, eval_ds = SLR53(train_pairs), SLR53(eval_pairs)
print("datasets ready")

In [ ]:
# 7. Collator + metrics (WER = word, CER = character)
from dataclasses import dataclass
import jiwer

@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labs = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        batch["labels"] = labs["input_ids"].masked_fill(labs.attention_mask.ne(1), -100)
        return batch

def compute_metrics(pred):
    pid, lid = pred.predictions, pred.label_ids
    lid = np.where(lid == -100, processor.tokenizer.pad_token_id, lid)
    hyp = processor.batch_decode(pid, skip_special_tokens=True)
    ref = processor.batch_decode(lid, skip_special_tokens=True)
    pr = [(r, h) for r, h in zip(ref, hyp) if r.strip()]
    if not pr: return {"wer": 100.0, "cer": 100.0}
    r, h = zip(*pr)
    return {"wer": 100 * jiwer.wer(list(r), list(h)),
            "cer": 100 * jiwer.cer(list(r), list(h))}

In [ ]:
# 8. Load Whisper + LoRA   (should take ~2 MINUTES. If it takes hours -> you have no GPU!)
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

model = WhisperForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.bfloat16).to("cuda")
model.generation_config.language = "bengali"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

In [ ]:
# 9. TRAIN  (checkpoints every 200 steps; auto-resumes if the session restarts)
import glob as _glob
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    warmup_steps=200,
    num_train_epochs=EPOCHS,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

cks = [c for c in _glob.glob(f"{CKPT_DIR}/checkpoint-*")
       if os.path.exists(os.path.join(c, "trainer_state.json"))]
cks.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = cks[-1] if cks else None
print("resuming from:", resume if resume else "scratch")

Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=Collator(processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
).train(resume_from_checkpoint=resume)

In [ ]:
# 10. FINAL HONEST SCORE on held-out clips (never trained on)
model.eval()
refs, hyps = [], []
for i, p in enumerate(eval_pairs):
    wav, _ = librosa.load(p["path"], sr=16000)
    feat = processor.feature_extractor(wav[: MAX_SEC * 16000], sampling_rate=16000,
                                       return_tensors="pt").input_features.to("cuda").to(torch.bfloat16)
    with torch.no_grad():
        ids = model.generate(feat, max_new_tokens=225)
    hyps.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
    refs.append(p["text"])
    if (i + 1) % 100 == 0: print(f"  {i+1}/{len(eval_pairs)}")

wer = 100 * jiwer.wer(refs, hyps)
cer = 100 * jiwer.cer(refs, hyps)
print("\n" + "=" * 62)
print(f"  HELD-OUT ({len(refs)} unseen clips, OpenSLR SLR53)")
print(f"  WER {wer:5.2f}%   ->  WORD accuracy      {100-wer:5.2f}%")
print(f"  CER {cer:5.2f}%   ->  CHARACTER accuracy {100-cer:5.2f}%")
print("=" * 62)

In [ ]:
# 11. Save the model (download it from the notebook's Output tab afterwards)
SAVE = "/kaggle/working/bengali_asr_final"
model.save_pretrained(SAVE)
processor.save_pretrained(SAVE)
print("saved ->", SAVE, "(grab it from the Output tab)")

## Reading the result

**Check `eval_wer` after epoch 1** (~3 hrs in) — it tells you early whether to keep going:

| eval_wer after epoch 1 | Meaning |
|---|---|
| **≤ 25%** | On track for ~90%. Let it run. |
| **25–40%** | Will land ~75–85%. Raise `N_PARTS` to 8–10 and rerun. |
| **> 40%** | Something's wrong — check GPU / that data loaded. |

## Levers if you fall short of 90%
1. **`N_PARTS = 10`** → the full 229 hours. **Biggest lever by far.**
2. `EPOCHS = 4-5` → only if eval_wer is still dropping each epoch.
3. Keep `MODEL = whisper-large-v3`.

**Never set EPOCHS=110.** Once eval_wer stops dropping, more epochs *overfit* — the training score improves while the honest held-out score gets worse.